# Introduction

Pour produire 1 kg d'acier, il faut utiliser entre 300 et 600 litres d'eau. L'eau est largement utilisée dans l'industrie pour refroidir les centrales, laver et alimenter les machines... Les usines consomment donc beaucoup d'eau dans leur processus de production.  
Or, les usines ont tendance à s'agglomérer autour de grands sites industriels pour des raisons d'efficacité.  
Nous pouvons donc nous demander quel est l'état des nappes phréatiques autour de ces grands sites industriels.

**Problématique** : dans quelle mesure l'exposition locale aux activités industrielles (ICPE) amplifie-t-elle la vulnérabilité et la baisse de niveau des nappes phréatiques ?

Derrière cette question s'en pose une autre, plus prospective : l'implantation des nouvelles méga-usines (batteries, hydrogène, data centers) se fait-elle de manière aveugle dans des bassins versants déjà condamnés à un épuisement de leurs nappes phréatiques ?

**Démarche.** Le niveau d'une nappe dépend de trois choses : la géographie, la pluie et l'industrie. Après la récupération des données (partie 1) et un premier croisement entre usines et stations (partie 2), on isole l'effet de la géographie (partie 3), puis celui du climat (partie 4), avant de mesurer celui de l'industrie (partie 5).

In [ ]:
%%capture
#Le rapport exécute les notebooks d'analyse sans afficher leurs sorties, puis n'affiche que les figures et tableaux choisis ci-dessous.
#eau_02 exécute eau_01 ; eau_04 exécute eau_03 (qui exécute eau_01) ; eau_05 réutilise eau_04 (chaque étape ne tourne qu'une fois)
import warnings
warnings.filterwarnings("ignore")
AFFICHER = False
%run ./eau_02_usines_stations.ipynb
%run ./eau_04_climat.ipynb
%run ./eau_05_industrie.ipynb
plt.close("all")
pd.set_option("display.precision", 2)

# 1. Données

Quatre sources ouvertes sont croisées, toutes téléchargées par API puis gardées en cache :

- **Géorisques (ICPE)** : les installations classées pour la protection de l'environnement. On ne garde que les installations industrielles « pures » (hors élevage, agriculture et éoliennes).
- **Hub'eau, niveaux des nappes** : les stations de mesure (piézomètres) et leurs séries de niveau entre 2015 et 2025.
- **Météo-France, réanalyse SAFRAN** : la pluie efficace (pluie moins évaporation) quotidienne sur une grille de 8 km.
- **Hub'eau, prélèvements (BNPE)** : le volume d'eau prélevé chaque année par chaque ouvrage, et son usage.

In [ ]:
resume_donnees = pd.DataFrame({
    "Jeu de données": ["Installations industrielles étudiées (ICPE)",
                       "Stations de mesure des nappes",
                       "Mesures de niveau 2015-2025",
                       "Stations suivies au moins 80 % des mois (parties 3 et 4)",
                       "Stations étudiées en partie 5 (celles de la partie 4)",
                       "Prélèvements souterrains déclarés (BNPE)"],
    "Effectif": [len(installations_indust), len(stations), len(chroniques), niveaux_mensuels.shape[1], len(st), len(sout)],
}).set_index("Jeu de données")
resume_donnees

## Pourquoi le Nord ?

In [ ]:
fig_classement_departements()

Le département le plus industrialisé compte environ 1 600 installations industrielles, loin devant le suivant (environ 1 200). Comme l'API Hub'eau fournit un très grand volume de données, on se concentre sur ce département : le Nord (59).

## Des mesures irrégulières

In [ ]:
fig_exemple_serie()

Les mesures ne tombent pas aux mêmes dates d'une station à l'autre, et certaines séries ont des trous de plusieurs mois (ici entre fin 2017 et mi-2019). On ramène donc chaque station à un niveau moyen par mois. Pour les parties 3 et 4, on ne garde que les stations mesurées au moins 80 % des mois de 2015-2025, en comblant par interpolation les trous de deux mois au plus.

# 2. Usines et stations : premier regard

In [ ]:
fig_carte_usines_stations()

Les usines (en rouge) se concentrent dans la métropole lilloise, sur le littoral dunkerquois, dans le bassin de Douai-Valenciennes et dans l'Avesnois. Les stations de mesure (en bleu) couvrent tout le département, avec une forte densité autour de Lille : de nombreuses stations ont des usines dans leur voisinage, ce qui rend la comparaison possible.

**Nappes reliées aux usines.** Pour chaque nappe, nombre d'usines situées à moins de 5 km d'au moins une de ses stations.

In [ ]:
nappes_usines[["code_nappe", "nom_nappe", "nb_usines"]].set_index("code_nappe")

Onze nappes sont reliées à au moins une usine. Les trois plus exposées, le calcaire carbonifère de Roubaix-Tourcoing, les sables du Landénien des Flandres et la craie de la vallée de la Deûle, ont chacune plus de 500 usines à moins de 5 km d'une de leurs stations.

In [ ]:
fig_evolution_par_pression()

En moyenne, les nappes qui ont plus de 5 usines à moins de 5 km ont un niveau inférieur d'environ 20 m. Ce n'est pas une preuve de causalité : le niveau d'une nappe est une altitude (par rapport au niveau de la mer), donc cet écart reflète d'abord l'endroit où se trouvent les stations. Les sauts brusques des courbes viennent aussi de ce que l'ensemble des stations mesurées change d'un mois à l'autre. Il faut donc d'abord neutraliser la géographie, puis le climat, avant de juger l'effet de l'industrie.

# 3. Conditions géographiques : altitude, roche, profondeur

Avant de chercher l'effet de l'industrie, on regarde si la géographie suffit à expliquer les **variations** des nappes : si deux nappes de même roche ou de même profondeur bougeaient de la même façon, il faudrait comparer les stations à géographie égale. L'ampleur des variations est mesurée par l'amplitude de chaque station (maximum − minimum du niveau mensuel sur 2015-2025).

## 3.1 Altitude

In [ ]:
fig_carte_altitude()

Les stations s'étagent du niveau de la mer (Flandre maritime, plaines de la Lys et de la Scarpe, métropole lilloise) à plus de 200 m dans l'Avesnois, au sud-est du département.

In [ ]:
fig_niveau_altitude()

Le niveau moyen suit l'altitude : la nappe est en général à quelques mètres sous le sol. Les exceptions, sous le niveau de la mer, sont des nappes **captives** (enfermées sous une couche imperméable) rabattues par des décennies de pompages : le calcaire carbonifère au nord de Lille (vers −60 m NGF) et les sables du Landénien sous l'argile des Flandres (Nieppe). On les garde dans l'analyse : leur baisse peut justement venir des prélèvements, industriels compris.

In [ ]:
fig_amplitude_altitude()

À première vue, l'amplitude monte avec l'altitude : la corrélation de Spearman est positive et significative (**ρ = 0,48**, p = 0,001, sur 45 stations). Mais le nuage n'a pas la forme d'une tendance : les faibles amplitudes (2 à 2,5 m) sont presque toutes serrées entre 15 et 20 m d'altitude, et au-delà de 100 m on trouve aussi bien 2 m que 17,6 m. Ce paquet de points bas est un groupe de stations particulier, que la roche permet d'identifier.

## 3.2 Roche

In [ ]:
fig_amplitude_roche()

In [ ]:
amplitude_par_roche

Les médianes vont de **2,3 m** (« Non renseignée ») à **3,9 m** (craie), mais l'écart *à l'intérieur* d'une roche est bien plus grand qu'entre roches : dans la craie, l'amplitude va de **1,7 m à 11 m** (19 stations), dans les calcaires de l'Avesnois de 2,9 à 17,6 m. Les autres familles n'ont que 3 à 5 stations, trop peu pour les départager. Le seul groupe nettement à part est « Non renseignée » : les **14 piézomètres** de 5 à 8 m de la plaine de la Scarpe, tous sous 3,7 m d'amplitude. Une nappe superficielle de plaine humide est drainée par les cours d'eau et les fossés dès qu'elle monte, elle reste donc dans une fourchette étroite (hypothèse non vérifiée).

## 3.3 Profondeur

In [ ]:
fig_amplitude_profondeur()

In [ ]:
correlations_amplitude

Sur les 45 stations, l'amplitude augmente avec la profondeur (**ρ = 0,40**, p = 0,007) et la médiane passe de 2,3 m (superficielles) à 3,2 m puis 3,5 m. Mais ce lien tient entièrement aux piézomètres de la Scarpe (en orange), tous à moins de 8 m et peu variables : sans eux, la corrélation tombe à **ρ = 0,09** (p = 0,62). Même chose pour l'altitude : sans la Scarpe, ρ = 0,32 (p = 0,08), ce n'est plus significatif au seuil de 5 %.

**En résumé**, la géographie fixe le **niveau moyen** d'une nappe, mais pas l'ampleur de ses **variations** : à roche ou profondeur égale, une station peut varier de 2 m comme de 11 m. Ces variations viennent d'abord de la météo, qu'il faut donc retirer.

# 4. Climat : retirer l'effet de la pluie

Pour chaque station, on explique le niveau mensuel par une régression linéaire sur la pluie efficace cumulée (sur 3 mois, 6 mois et une « mémoire » de k = 12, 24 ou 36 mois), un terme saisonnier et une tendance dans le temps :

$$\text{niveau}_t = a + b \, PE_{k \text{ derniers mois}} + c \, PE_{3 \text{ derniers mois}} + e \, PE_{6 \text{ derniers mois}} + \text{saison (sinus, cosinus)} + d \, \text{temps}$$

On retient pour chaque station la mémoire k qui donne le meilleur R². On retire ensuite au niveau observé la partie climatique (constante + pluie + saison) : le résidu garde la tendance $d \times t$ et tout ce que le climat n'explique pas.

## 4.1 Le modèle est-il cohérent ?

In [ ]:
fig_memoire_profondeur()

Plus l'ouvrage est profond, plus la mémoire retenue est longue : la part des stations à k = 12 mois passe de 56 % (superficielles) à 18 % (intermédiaires) puis 14 % (profondes). C'est le comportement attendu, les nappes profondes étant plus inertes. C'est un contrôle de cohérence, pas une preuve, car k ne prend que 3 valeurs et les R² de deux k voisins sont souvent proches.

In [ ]:
fig_cycle_saisonnier()

Dans les deux graphiques, les points montent de gauche à droite : un hiver plus pluvieux que d'habitude fait davantage remonter la nappe, un été plus pluvieux la fait moins baisser. Le cycle change donc d'une année à l'autre avec la pluie, alors qu'un pompage régulier donnerait à peu près le même chaque année : il vient surtout du climat, et le retirer est justifié. Réserve : les étés secs sont aussi ceux où l'on pompe le plus, ce qui peut accentuer la baisse d'été.

## 4.2 Tendances une fois le climat retiré

In [ ]:
fig_pentes_brute_corrigee()

In [ ]:
resume_tendances

Toutes les stations sont sous la diagonale : pour chacune, la pente corrigée est plus basse que la pente brute. Sur la période, la pluie a donc poussé les nappes vers le haut (années humides en fin de période, dont la recharge record de 2023-2024), ce qui masque une partie de la baisse dans le niveau observé. Avant correction, les stations se partagent entre baisses (26) et hausses (19), avec une médiane proche de zéro (−0,6 cm/an). Après correction, 41 stations sur 45 sont en baisse, avec une médiane de −4,6 cm/an : à pluie égale, la plupart des nappes baissent légèrement.

Les trois plus fortes baisses corrigées sont des nappes captives (Bondues, Nieppe, Quesnoy-sur-Deûle), pour lesquelles le modèle, qui suppose une recharge par la pluie locale, est à lire avec prudence. C'est cette tendance corrigée que la partie 5 compare à la pression industrielle.

**Limite.** Le modèle sous-estime probablement l'effet humain. Les prélèvements sont plus forts en été et les années sèches ; or le modèle a justement deux termes qui varient de la même façon, la saison et la pluie efficace. Tout ce que les prélèvements ont de saisonnier ou de lié à la sécheresse est donc compté comme « climat » et sort du résidu.

# 5. Effet de l'industrie sur les nappes phréatiques

On cherche si les nappes situées près des installations industrielles ont évolué plus défavorablement que les autres, **une fois l'effet de la pluie retiré**. L'évolution de chaque station est sa tendance corrigée du climat (partie 4), en cm/an : elle utilise toute la série, ne dépend donc pas du choix des périodes, et une année exceptionnellement humide comme 2024 ne la fausse plus.

## 5.1 Tendance des nappes, climat retiré

In [ ]:
fig_distribution_evolutions()

Une fois la pluie retirée, 41 des 45 stations sont en baisse, avec une médiane de −4,6 cm/an : à pluie égale, la nappe typique du Nord baisse légèrement. La moitié des stations ont une tendance comprise entre −11,8 et −1,9 cm/an, et les 4 stations en hausse le sont de moins de 4 cm/an. Les plus fortes baisses sont Bondues (−57 cm/an), Nieppe (−36 cm/an) et Quesnoy-sur-Deûle (−24 cm/an) : trois nappes captives, pour lesquelles le modèle est à lire avec prudence.

In [ ]:
fig_plus_fortes_evolutions()

> **À compléter après exécution.** Décrire les trajectoires : les baisses sont-elles régulières ou concentrées sur quelques années ? Bondues se distingue-t-elle des autres ?

## 5.2 Premier indicateur de pression : le nombre d'usines

On ne garde que l'industrie manufacturière (codes NAF 10 à 33) et l'énergie (code 35), les activités les plus consommatrices d'eau, et on compte les usines à moins de 2, 5 et 10 km de chaque station.

In [ ]:
fig_evolution_par_pression_boxplot()

In [ ]:
evolution_par_pression[["count", "50%", "mean", "std"]].rename(columns={"count": "stations", "50%": "médiane (cm/an)", "mean": "moyenne (cm/an)", "std": "écart-type"})

> **À compléter après exécution.** Le groupe « plus de 5 usines » baisse-t-il plus que les autres ? Commenter la médiane plutôt que la moyenne, tirée par quelques stations extrêmes, et l'effectif de chaque groupe.

In [ ]:
fig_carte_evolutions()

> **À compléter après exécution.** Les stations en baisse (rouge) se concentrent-elles autour des pôles industriels ? La couleur sature à ±10 cm/an.

## 5.3 Ce qui distingue les stations en baisse

In [ ]:
fig_evolution_geologie_profondeur()

> **À compléter après exécution.** Comparer l'écart entre les médianes selon la géologie et selon la profondeur à celui selon la pression industrielle. Vérifier si les stations profondes sont aussi les calcaires, auquel cas on ne peut pas séparer les deux effets.

## 5.4 Deuxième indicateur de pression : les volumes pompés

Le nombre d'usines est un indicateur imparfait : une petite et une grande usine comptent pareil, et une usine ne pompe pas forcément dans la nappe. On utilise donc aussi les volumes réellement prélevés, déclarés dans la BNPE.

In [ ]:
fig_part_usages()

In [ ]:
fig_prelevements_par_an()

In [ ]:
comparaison.round(1)

L'eau potable domine très largement les prélèvements en eau souterraine : 79,5 % des volumes entre 2015 et 2023, contre 18,2 % pour l'industrie et 2,3 % pour l'irrigation. Les prélèvements pour l'eau potable baissent d'environ 8 % entre 2015-2017 et 2021-2023, ceux de l'industrie restent stables (+3 %), et ceux de l'irrigation, très faibles, sont plus élevés les années sèches (2020 et 2022). L'industrie pompe donc peu dans les nappes du Nord, et de façon stable.

In [ ]:
fig_carte_prelevements()

Les prélèvements pour l'eau potable (bleu) sont les plus importants et se concentrent autour de la métropole lilloise, de Douai-Valenciennes et de l'Avesnois. Les prélèvements industriels (noir) sont dispersés et généralement plus petits. L'irrigation (vert) correspond à de nombreux petits prélèvements en Flandre intérieure et dans le Cambrésis.

**Prélèvements autour des stations clés** (volumes annuels moyens à moins de 5 km, en millions de m³, et tendance corrigée du climat).

In [ ]:
focus.set_index("code_bss")

In [ ]:
fig_bondues_lille()

- **Nieppe** (−36 cm/an) : quasiment aucun prélèvement (0,01 million de m³, uniquement pour l'irrigation). Sa baisse ne vient pas d'un pompage local, encore moins de l'industrie.
- **Bondues** (−57 cm/an) : 5,8 millions de m³ par an, dont 36 % pour l'industrie. Le pompage y est important, mais l'eau potable y pèse davantage que l'industrie.
- **Lille** : une fois la pluie retirée, les cinq piézomètres sont presque stables, entre −4,6 et +2,7 cm/an, alors qu'ils sont entourés de 3,3 à 7,0 millions de m³ de prélèvements annuels. Deux d'entre eux ont plus de prélèvements totaux que Bondues, et deux autres plus de prélèvements industriels.

À pompage comparable, voire supérieur, les nappes de Lille perdent au plus quelques centimètres par an, quand celle de Bondues perd plus d'un demi-mètre par an : ce n'est pas la quantité d'eau prélevée qui fait la différence. Limite : la BNPE n'indique pas dans quelle nappe pompe chaque forage.

**Corrélation entre la tendance des nappes et la pression humaine** (Spearman), après et avant retrait de la pluie. Une corrélation négative indiquerait que les nappes les plus exposées baissent davantage.

In [ ]:
tableau_corr.set_index("Indicateur de pression")

> **À compléter après exécution.** Les corrélations avec la tendance corrigée sont-elles négatives ? Fortes ou faibles ? Le résultat change-t-il selon le rayon, l'indicateur, et par rapport à la tendance brute ? Une corrélation ne prouve pas une cause.

# Conclusion générale

**La géographie fixe le niveau des nappes, pas leurs variations.** Le niveau moyen d'une nappe suit l'altitude de sa station, sauf pour quatre nappes captives rabattues sous le niveau de la mer. L'ampleur des variations, elle, ne s'explique ni par l'altitude, ni par la roche, ni par la profondeur, dès que l'on met à part les piézomètres superficiels de la Scarpe (partie 3).

**Le climat pilote l'essentiel des variations, et masque une légère baisse.** Le cycle saisonnier de chaque nappe varie d'une année à l'autre avec la pluie de la saison (partie 4). Une fois la part de la pluie retirée, 41 stations sur 45 baissent, avec une médiane de −4,6 cm/an : les années humides de la fin de période masquaient cette baisse dans le niveau observé (parties 4 et 5).

**L'industrie est un préleveur minoritaire.** Elle ne représente que 18 % des prélèvements souterrains, contre 80 % pour l'eau potable, et ses volumes sont restés stables. À pompage comparable, Bondues perd plus d'un demi-mètre par an quand les piézomètres de Lille sont presque stables (partie 5).

> **À compléter après exécution.** Le lien entre la tendance corrigée du climat et la pression industrielle (partie 5), puis la réponse à la problématique : l'exposition locale aux activités industrielles amplifie-t-elle la baisse des nappes, et quel conseil en tirer pour l'implantation de nouvelles usines très consommatrices d'eau ?

**Limites.**
- Un échantillon réduit (45 piézomètres bien suivis), avec de petits effectifs dans certains groupes ; la géologie et la profondeur se recoupent.
- Des corrélations, qui ne prouvent pas une causalité.
- Le modèle climat sous-estime probablement l'effet humain : ce que les prélèvements ont de saisonnier ou de lié à la sécheresse est compté comme « climat ». Les plus fortes baisses sont des nappes captives, pour lesquelles il est à lire avec prudence.
- La BNPE n'indique pas dans quelle nappe pompe chaque forage.

**Perspective.** Reproduire l'analyse dans une région plus sèche ou dont l'industrie consomme davantage d'eau.